In [ ]:
# input
potential_file = "../data/human_potential_258.tsv"
align_dir = "./tmp/clustalo/"
metaldb_pred = "../../../predict_afdb/data/pred_ge_3_clique_3.tsv"
# output
high_conserved_pros = "./data/human_res_consistency_0.8.tsv"

In [2]:
from Bio import AlignIO
from Bio.SeqRecord import SeqRecord
import os

def get_alignments(file: str):
    
    if not os.path.exists(file):
        return []
    try:
        alignment = AlignIO.read(file, "clustal")
    except:
        alignment = []
    return list(alignment)


def get_species_cov(alignments: list[str], positions: set[int]):

    def real_posi_to_aligned_posi(seq: str):
        result = dict()
        real_posi = -1
        for idx, aa in enumerate(seq):
            if aa != "-":
                real_posi += 1
                result[real_posi] = idx

        return result

    if len(alignments) == 0:
        return {
            "pro_cov": .0,
            "res_cov": .0
        }
    
    assert len(alignments) >= 2

    human_aln: SeqRecord = alignments[0]
    human_seq = human_aln.seq.replace("-", "")
    posi_dict = real_posi_to_aligned_posi(human_aln)
    ref_aligned_posi_to_resi = dict()
    for p in positions:
        resi = human_seq[p]
        aligned_posi = posi_dict[p]
        ref_aligned_posi_to_resi[aligned_posi] = resi

    matched_cnt = 0
    targets = alignments[1:]
    for t in targets:
        is_matched = True
        for p, r in ref_aligned_posi_to_resi.items():
            if t[p] != r:
                is_matched = False
                break
        if is_matched: matched_cnt += 1

    pro_cov = len(targets) / 10 # 11 species except human itself
    res_cov = matched_cnt / 10

    return {
        "pro_cov": pro_cov,
        "res_cov": res_cov
    }

In [ ]:
import pandas as pd


metal_db_ids = set(pd.read_table(metaldb_pred)['seq_id'].map(lambda x: x.split("-")[1]))

In [18]:
df = pd.read_table(potential_file)

In [23]:
records = []
for _, row in df.iterrows():
    seq_id = row['seq_id']
    pred_num_resi = row['pred_num_resi'].split(",")
    pred_num_posi = [int(i[:-1]) - 1 for i in pred_num_resi]

    aln_file = f"{align_dir}/{seq_id}/{seq_id}.clu"
    aligns = get_alignments(aln_file)

    if len(aligns) == 0:
        pred_cov = 0
    else:
        targets = aligns[1:]
        pred_cnt = 0
        for a in targets:
            if a.id in metal_db_ids:
                pred_cnt += 1
        pred_cov = pred_cnt / len(targets)

    result = get_species_cov(aligns, set(pred_num_posi))
    records.append({
        "seq_id": seq_id,
        "pred_cov": pred_cov,
        **result
    })

In [ ]:
df_sel = pd.merge(df, pd.DataFrame(records), on="seq_id").sort_values(by=['res_cov','pro_cov'], ascending=False)
df_sel[df_sel['res_cov'] >= 0.8].to_csv(high_conserved_pros, sep="\t", index=None)

In [26]:
df_sel[df_sel['res_cov'] >= 0.8]

,seq_id,rep_id,pred_num_resi,proba,plddt,metal_type,metal_group_type,len,name,gene_name,pred_cov,pro_cov,res_cov
142,Q9HB07,Q9HB07,"50H,55H,57D,93D,107H,108H,176D","0.8565,0.2281,0.3082,0.7336,0.7544,0.549,0.2898","98.11,98.38,98.64,98.64,98.11,97.06,98.11","3,0,2,3,0,0,2","1,1,1,1,1,1,1",376,MYG1 exonuclease,MYG1,0.900,1.0,0.9
83,Q9H5J4,A0A673GDS4,"144H,145H,174H","0.7772,0.9395,0.5965","97.35,97.58,98.03","0,0,0","1,1,1",265,Elongation of very long chain fatty acids prot...,ELOVL6,1.000,0.8,0.8
84,Q9HB03,Q9HB03,"127E,148H,149H,178H","0.2243,0.7548,0.9328,0.6065","97.37,96.62,97.12,97.61","5,0,0,5","1,1,1,1",270,Elongation of very long chain fatty acids prot...,ELOVL3,1.000,0.8,0.8
85,Q9H2C2,A0A3Q0FWI8,"34C,37C,58C,61C","0.9798,0.9574,0.9218,0.9477","95.13,92.68,93.77,94.58","0,0,0,0","1,1,1,1",271,Protein ARV1,ARV1,1.000,0.8,0.8
96,Q9BW60,Q9JLJ5,"144H,145H,175H","0.3712,0.7343,0.7068","95.27,96.11,97.5","5,0,5","1,1,1",279,Elongation of very long chain fatty acids prot...,ELOVL1,1.000,0.8,0.8
97,A1L3X0,G3TBW4,"150H,151H,181H","0.3263,0.7727,0.7058","94.88,95.44,96.82","5,0,5","1,1,1",281,Elongation of very long chain fatty acids prot...,ELOVL7,1.000,0.8,0.8
103,Q9NXB9,Q9JLJ4,"149H,150H,180H","0.36,0.8302,0.6816","96.71,96.42,97.85","0,0,0","1,1,1",296,Elongation of very long chain fatty acids prot...,ELOVL2,1.000,0.8,0.8
107,Q9NYP7,Q9NYP7,"146H,147H,177H","0.3736,0.8476,0.6919","95.12,94.29,96.51","0,0,0","1,1,1",299,Elongation of very long chain fatty acids prot...,ELOVL5,1.000,0.8,0.8
108,Q9BXY0,A0A1A7YQ79,"17C,29C,39C,44C","0.8488,0.9458,0.9835,0.9642","92.05,94.03,92.79,92.79","0,0,0,0","1,1,1,2",300,Protein MAK16 homolog,MAK16,1.000,0.8,0.8
116,P49247,A0A481BC81,"160D,163D,182E","0.2897,0.5349,0.4342","98.63,98.36,98.63","3,1,2","0,1,1",311,Ribose-5-phosphate isomerase,RPIA,1.000,0.8,0.8
